# 02 — Feature Engineering & Frequency Alignment
Align daily + monthly data, build the feature matrix for all models.


In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120

## 1. Load raw data

In [19]:
prices = pd.read_csv('../data/raw/daily_prices.csv', index_col=0, parse_dates=True)
macro  = pd.read_csv('../data/raw/monthly_macro.csv', index_col=0, parse_dates=True)

prices.index = prices.index.tz_localize(None)  # strip tz if present
print('Daily shape:', prices.shape)
print('Monthly shape:', macro.shape)

Daily shape: (2854, 8)
Monthly shape: (2991, 6)


## 2. Daily features

In [20]:
df = pd.DataFrame(index=prices.index)

# Target
df['silver']         = prices['silver']
df['silver_return']  = np.log(prices['silver']).diff()   # log-return

# Price-based covariates
df['gold_return']    = np.log(prices['gold']).diff()
df['copper_return']  = np.log(prices['copper']).diff()
df['usd_return']     = np.log(prices['usd_index']).diff()
df['sp500_return']   = np.log(prices['sp500']).diff()
df['gs_ratio']       = prices['gold'] / prices['silver']  # gold/silver ratio (level)

# Static z-score using training-period statistics only (no look-ahead).
# The ratio has hovered around ~80 for the entire sample, so a fixed historical
# baseline captures "stretched vs the long-run norm" better than a rolling window
# (which drifts toward the spike during prolonged deviations like Mar 2020).
# Train period = pre-2022; the same mean/std are applied to val and test.
_GS_TRAIN_MASK = df.index < '2022-01-01'
_GS_MEAN = df.loc[_GS_TRAIN_MASK, 'gs_ratio'].mean()
_GS_STD  = df.loc[_GS_TRAIN_MASK, 'gs_ratio'].std()
df['gs_ratio_z'] = (df['gs_ratio'] - _GS_MEAN) / _GS_STD
print(f'gs_ratio_z: train-period mean={_GS_MEAN:.2f}, std={_GS_STD:.2f}')

df['vix_return']     = np.log(prices['vix']).diff()
df['oil_return']     = np.log(prices['oil']).diff()

# Lags of silver return (AR terms for ARIMAX)
for lag in [1, 2, 3, 5, 10]:
    df[f'silver_lag{lag}'] = df['silver_return'].shift(lag)

# Rolling volatility (realised vol proxy)
df['silver_vol_5d']  = df['silver_return'].rolling(5).std()
df['silver_vol_21d'] = df['silver_return'].rolling(21).std()

# Rolling momentum
df['mom_5d']         = df['silver_return'].rolling(5).sum()
df['mom_21d']        = df['silver_return'].rolling(21).sum()

print(df.shape)
df.head()

gs_ratio_z: train-period mean=78.50, std=8.89
(2854, 19)


/Users/asier.ugartechegmail.com/miniforge3/envs/tf/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,silver,silver_return,gold_return,copper_return,usd_return,sp500_return,gs_ratio,gs_ratio_z,vix_return,oil_return,silver_lag1,silver_lag2,silver_lag3,silver_lag5,silver_lag10,silver_vol_5d,silver_vol_21d,mom_5d,mom_21d
Date,,,,,,,,,,,,,,,,,,,
2015-01-02,15.734000,NaN,NaN,NaN,NaN,NaN,75.378161,-0.350826,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,16.179001,0.027890,0.014980,-0.016159,0.003288,-0.018447,74.411271,-0.459633,0.113088,-0.051603,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-06,16.603001,0.025869,0.012711,0.003931,0.001312,-0.008933,73.438535,-0.569098,0.058496,-0.043081,0.027890,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-07,16.510000,-0.005617,-0.007161,-0.002857,0.004253,0.011563,73.325255,-0.581845,-0.089597,0.014910,0.025869,0.027890,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-08,16.351000,-0.009677,-0.001819,0.003926,0.005210,0.017730,73.903739,-0.516747,-0.126822,0.002874,-0.005617,0.025869,0.02789,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Align monthly macro to daily (forward-fill)
Forward-fill is the correct approach: a monthly CPI reading published on day T is known for all days until the next release.

In [12]:
# Reindex monthly macro to daily, then forward-fill
macro_daily = macro.reindex(df.index, method='ffill')

# Add month-over-month changes (stationary version of levels)
for col in macro.columns:
    macro_daily[f'{col}_chg'] = macro[col].pct_change().reindex(df.index, method='ffill')

df = pd.concat([df, macro_daily], axis=1)
print('Combined shape:', df.shape)

Combined shape: (2854, 30)


## 3b. COT positioning — COMEX silver disaggregated report

Weekly Commitments-of-Traders data from the CFTC (`disaggregated_fut` report,
collected via `src/collect_cot.py` → `data/raw/cot_silver.csv`). Two derived series:

- `cot_mm_net_pct`   = (Managed Money longs − shorts) / Open Interest — speculative positioning
- `cot_comm_net_pct` = (Producer/Merchant longs − shorts) / Open Interest — hedger positioning

**Lag rigour**: CFTC publishes Friday afternoon with data as-of the preceding Tuesday.
To predict silver at Friday $t$ we need features knowable by Friday $t-1$ close. The
report dated Friday $t-1$ carries Tuesday $t-1$ data — fully observed before our target,
so a 1-week lag in W-FRI weekly notebooks is leak-safe. Here we just forward-fill from
the report date onto the daily index — downstream notebooks add the lag.

In [13]:
import os
cot_path = '../data/raw/cot_silver.csv'
if not os.path.exists(cot_path):
    print('cot_silver.csv missing — run `python src/collect_cot.py` first.')
else:
    cot = pd.read_csv(cot_path, parse_dates=['report_date']).sort_values('report_date')
    cot = cot.set_index('report_date')

    mm_net   = cot['M_Money_Positions_Long_All']  - cot['M_Money_Positions_Short_All']
    comm_net = cot['Prod_Merc_Positions_Long_All'] - cot['Prod_Merc_Positions_Short_All']
    oi       = cot['Open_Interest_All']

    cot_feats = pd.DataFrame({
        'cot_mm_net_pct':   mm_net   / oi,
        'cot_comm_net_pct': comm_net / oi,
    })

    # Reindex onto daily, forward-fill so each business day after a release inherits
    # that release's snapshot. The forward-fill mirrors monthly_macro treatment.
    cot_daily = cot_feats.reindex(df.index, method='ffill')

    df = pd.concat([df, cot_daily], axis=1)
    print(f'COT merged. Range: {cot.index.min().date()} → {cot.index.max().date()}')
    print('Daily coverage (after ffill):')
    print(df[['cot_mm_net_pct','cot_comm_net_pct']].notna().mean().round(3))


COT merged. Range: 2006-06-13 → 2026-05-05
Daily coverage (after ffill):
cot_mm_net_pct      1.0
cot_comm_net_pct    1.0
dtype: float64


## 4. Add sentiment (placeholder — fill in after 06_sentiment.ipynb)

In [14]:
import os
sentiment_path = '../data/processed/daily_sentiment.csv'
if os.path.exists(sentiment_path):
    sentiment = pd.read_csv(sentiment_path, index_col=0, parse_dates=True)
    df = df.join(sentiment, how='left')
    print('Sentiment joined:', sentiment.columns.tolist())
else:
    print('No sentiment file yet — run 06_sentiment.ipynb first.')
    df['sentiment_score'] = np.nan

Sentiment joined: ['reddit_sentiment', 'reddit_weight_sum', 'reddit_post_count', 'news_sentiment', 'news_article_count', 'sentiment_score']


## 5. Google Trends — retail attention index

Weekly search volume for silver-related terms ("silver", "buy silver", "silver price").
Fetched via `src/collect_trends.py` using pytrends, stitched across overlapping 5-year
windows and forward-filled to daily frequency.

Reference: Da, Engelberg & Gao (2011) *In Search of Attention*, Journal of Finance.

# TODO: Add COT (Commitment of Traders) report from CFTC as an additional sentiment/positioning
# feature. Weekly speculative net positions in silver futures — a direct measure of
# speculative sentiment from actual market participants (not social media).
# Source: https://www.cftc.gov/MarketReports/CommitmentsofTraders/index.htm
# Package: pip install cot-reports  or  fetch directly from CFTC CSV downloads.


In [15]:
trends_path = '../data/raw/google_trends.csv'
if os.path.exists(trends_path):
    trends = pd.read_csv(trends_path, index_col=0, parse_dates=True)
    trends.index = trends.index.tz_localize(None)

    # Use the composite index column; forward-fill weekly -> daily
    trends_daily = trends['trends_silver'].reindex(df.index, method='ffill')

    # Normalise to 0-1 so it's on the same scale as other features
    df['trends_silver'] = trends_daily / 100.0

    # 4-week rolling mean — smooths the weekly steps for daily models
    df['trends_silver_ma4'] = df['trends_silver'].rolling(28).mean()

    print(f'Google Trends added. Coverage: {trends_daily.first_valid_index().date()} '
          f'-> {trends_daily.last_valid_index().date()}')
    print(f'NaNs before first valid: {trends_daily.isna().sum()}')
else:
    print('google_trends.csv not found — run src/collect_trends.py first.')
    df['trends_silver']     = 0.0
    df['trends_silver_ma4'] = 0.0


Google Trends added. Coverage: 2015-01-02 -> 2026-05-04
NaNs before first valid: 0


## 6. Train / validation / test split
- Train: 2015–2021
- Val:   2022
- Test:  2023–2024


In [21]:
df = df.dropna(subset=['silver_return'])

train = df[df.index < '2022-01-01']
val   = df[(df.index >= '2022-01-01') & (df.index < '2023-01-01')]
test  = df[df.index >= '2023-01-01']

print(f'Train: {train.index[0].date()} → {train.index[-1].date()}  ({len(train)} days)')
print(f'Val:   {val.index[0].date()} → {val.index[-1].date()}  ({len(val)} days)')
print(f'Test:  {test.index[0].date()} → {test.index[-1].date()}  ({len(test)} days)')

Train: 2015-01-05 → 2021-12-31  (1755 days)
Val:   2022-01-03 → 2022-12-30  (251 days)
Test:  2023-01-03 → 2026-05-04  (837 days)


## 7. Save feature matrix

In [22]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/features.csv')
train.to_csv('../data/processed/train.csv')
val.to_csv('../data/processed/val.csv')
test.to_csv('../data/processed/test.csv')

print('Saved feature matrix and splits to data/processed/')

Saved feature matrix and splits to data/processed/
